# Load the model that will generate qa pairs

In [1]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="deepseek-ai/DeepSeek-R1-Distill-Qwen-32B",
    max_seq_length=1024,
    load_in_4bit=True,
    device_map="auto",  # Auto-map to GPU
)

# Enable native 2x faster inference
model = FastLanguageModel.for_inference(model)

print("Loaded model in 4-bit ✅")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
INFO 10-06 17:12:08 [__init__.py:216] Automatically detected platform cuda.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.9.11: Fast Qwen2 patching. Transformers: 4.56.2. vLLM: 0.10.2.
   \\   /|    NVIDIA RTX A5000. Num GPUs = 1. Max memory: 23.673 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33+5146f2a.d20251002. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Loaded model in 4-bit ✅


# Load the corpus

In [2]:
import os 

#CORPUS_DIR = "/storage/corpus/wtk_archive_with_stops"
CORPUS_DIR = "/storage/corpus/ai_corpus_slimmer_clean"

BLOCK_SIZE = 1024  # max tokens per chunk

tok = tokenizer 

# Ensure EOS/PAD exist and are consistent
added = False
if tok.eos_token is None:
    tok.add_special_tokens({"eos_token": "</s>"})
    added = True
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
    added = True
if added:
    model.resize_token_embeddings(len(tok))

# -----------------------
# 2) Load raw text files (no EOS strings here)
# -----------------------
def load_txt_corpus(directory):
    texts = []
    for filename in os.listdir(directory):
        if not filename.endswith(".txt"):
            continue
        path = os.path.join(directory, filename)
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            txt = f.read().strip()
            if txt:
                texts.append(txt)
    return texts

raw_texts = load_txt_corpus(CORPUS_DIR)
print(f"Loaded {len(raw_texts)} files from {CORPUS_DIR} ✅")

Loaded 11882 files from /storage/corpus/ai_corpus_slimmer_clean ✅


# Build the training chunks

In [29]:
from typing import List, Optional

def split_by_eos_or_chunk(
    text: str,
    tokenizer,
    min_tokens: int = 20,
    max_segments: Optional[int] = None,
    approx_token_budget: Optional[int] = 25_000,
    alt_markers: Optional[List[str]] = None,
) -> List[str]:
    """
    Prefer splitting on tokenizer's EOS token (string or id).
    If no EOS markers are present, fall back to token-length chunking.
    Ensures segments have at least `min_tokens` and stops around desired volume.
    """
    alt_markers = alt_markers or ["</s>", "<|im_end|>", "<|endoftext|>"]

    # 1) Try string-based split using known markers (fast path)
    s = text
    for m in alt_markers:
        if m and m in s:
            parts = [p.strip() for p in s.split(m)]
            # filter by token count, not chars
            kept = []
            for p in parts:
                if not p:
                    continue
                if len(tokenizer.encode(p, add_special_tokens=False)) >= min_tokens:
                    kept.append(p)
            if kept:
                # enforce target volume and segment cap
                if approx_token_budget:
                    total = 0
                    budgeted = []
                    for seg in kept:
                        n = len(tokenizer.encode(seg, add_special_tokens=False))
                        if total + n > approx_token_budget or (max_segments and len(budgeted) >= max_segments):
                            break
                        budgeted.append(seg)
                        total += n
                    return budgeted
                if max_segments:
                    return kept[:max_segments]
                else:
                    return kept

    # 2) Fallback: chunk by tokens to ~fixed size if no markers found
    ids = tokenizer.encode(text, add_special_tokens=False)
    # choose a chunk size that’s friendly for your model/context
    CHUNK_TOKENS = max(min_tokens * 4, 1024)  # e.g., 512–1024
    segments = []
    i = 0
    total = 0
    while i < len(ids) and (approx_token_budget is None or total < approx_token_budget) and (not max_segments or len(segments) < max_segments):
        j = min(i + CHUNK_TOKENS, len(ids))
        seg_text = tokenizer.decode(ids[i:j], skip_special_tokens=True).strip()
        if seg_text:
            segments.append(seg_text)
            total += (j - i)
        i = j
    return segments

blocks = []
for text in raw_texts:
    segments = split_by_eos_or_chunk(
        text,
        tokenizer,
        min_tokens=50,
        max_segments=50,
        approx_token_budget=25_000,           # aim for ~25k tokens total
        alt_markers=["</s>", "<|im_end|>"],   # add any markers your data uses
    )
    blocks.extend(segments)
    if len(blocks) % 1000 == 0:
        print(f"Processed {len(blocks)} number of blocks...")
    #print(f"Procssed raw text with len {len(text)} which created len segments {len(segments)}")

print(f"Prepared {len(blocks)} packed training chunks ✅")


Processed 6000 number of blocks...
Processed 9000 number of blocks...
Processed 10000 number of blocks...
Processed 12000 number of blocks...
Processed 13000 number of blocks...
Processed 15000 number of blocks...
Processed 16000 number of blocks...
Processed 18000 number of blocks...
Processed 21000 number of blocks...
Processed 24000 number of blocks...
Prepared 25493 packed training chunks ✅


# Generate qa pairs

In [ ]:
SEGMENTS_LIMIT = 100000
MAX_TOKENS = 512
REPORTING_INTERVAL = 10
JSON_WRITE_INTERVAL = 500

qa_pairs = []

import json
import re

print(f"Generating qa pairs for {len(blocks)} number of blocks...")
      
i = 0
total_pairs = 0
for segment in blocks:
    i = i + 1
    if i > SEGMENTS_LIMIT:
        print(f"Hit segments limit {SEGMENTS_LIMIT}. Stopping.")
        break

    if i % REPORTING_INTERVAL == 0:
        print(f"Processed {i} number of segments...")

    # Truncate to ~512-1024 tokens for context length (align with max_seq_length=1024)
    tokens = segment.split()[:1024]
    segment = " ".join(tokens)

    # Prompt with escaped curly braces for JSON example
    prompt = f"""From this text segment: {segment}

Generate 5 diverse Q-A pairs:
- 2 factual (e.g., who/what/when/where).
- 1 explanatory (e.g., why/how).
- 1 inference-based (e.g., what might happen next?).
- 1 domain-specific (e.g., if code, explain function; if news, key implications).
Ensure questions are specific, non-generic, and answers are concise (<250 words), directly extracted/inferred from the segment.
Include </s> at the end of each answer to match corpus format. /no_think
Format as JSON list: [{{ "question": "...", "answer": "...</s>"}}, ...]"""

    try:
        # Tokenize and generate
        inputs = tokenizer(prompt, return_tensors="pt", max_length=1024, truncation=True).to("cuda")
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_TOKENS,  # Limit for speed
            do_sample=False,  # Deterministic for consistency
            num_return_sequences=1,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        #print(f"Response: {response}")

        # Parse JSON (trim prompt and handle </s>)
        think_start = response.find("[")
        json_start = response.find("[", think_start + 1)
        json_end = response.rfind("]", json_start + 1) + 1

        if json_start != -1 and json_end != -1:

            snippet = response[json_start:json_end]
            qa_list = json.loads(snippet)

            # Ensure </s> in answers
            for qa in qa_list:
                if not qa["answer"].endswith("</s>"):
                    qa["answer"] += "</s>"
                qa_pairs.append(qa)
                total_pairs = total_pairs + 1
    except Exception as e:
        print(f"Error processing segment {i}: {e}")
        continue

     # Save Q-A pairs to JSON for LoRA training on interval
    if i > 0 and i % JSON_WRITE_INTERVAL  == 0:
        output_filename = "qa_pairs_" + str(i) + ".json"
        with open(output_filename, "w", encoding="utf-8") as f:
            json.dump(qa_pairs, f, ensure_ascii=False, indent=2)
        print(f"Wrote {len(qa_pairs)} qa pairs to file {output_filename}...")
        qa_pairs = []

print(f"Generated {len(total_pairs)} total QA pairs")

Generating qa pairs for 25493 number of blocks...
Processed 10 number of segments...
